# 03 — Phase 3: Finding the Arbitration Heads

This is the **core original contribution** of the paper.

We test the hypothesis that a subset of the 144 attention heads:
1. Have significantly higher activation magnitude during conflict vs. unambiguous
2. Causally delay or prevent conflict resolution when ablated
3. Are *not* strongly part of either Circuit A or Circuit B (novel heads)
4. Generalise across multiple prompt categories

**Steps**
1. Compute activation magnitude delta (conflict − unambiguous) for all 144 heads
2. Ablation test: zero each top-candidate head, measure flip rate
3. Activation patching test: patch losing-circuit activations into each candidate
4. Cross-category overlap: Jaccard similarity of top-10 candidates

**Outputs**
- `data/results/phase3/activation_delta.npy`
- `data/results/phase3/ablation_results.csv`
- `data/results/phase3/patch_results.csv`
- `data/results/phase3/jaccard_matrix.csv`

**Expected runtime:** ~4–8 hours (ablation sweep over 144 heads × all prompts)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from circuit_conflict.utils import load_model, get_device, get_answer_token_id
from circuit_conflict.dataset import load_prompts
from circuit_conflict.patching import (
    head_activation_magnitudes,
    ablate_head_logit_diff,
    ablate_head_layer_curve,
    ablation_flip_rate,
    patch_losing_circuit_into_head,
)
from circuit_conflict.metrics import (
    arbitration_delta_score,
    rank_heads_by_delta,
    cross_category_jaccard,
    top_k_heads,
)

DEVICE = get_device()
RESULTS_DIR   = Path('../data/results/phase3')
PHASE1_DIR    = Path('../data/results/phase1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
model = load_model(DEVICE)
N_LAYERS = model.cfg.n_layers  # 12
N_HEADS  = model.cfg.n_heads   # 12

df_all      = load_prompts()
df_conflict = df_all[df_all.is_conflict].reset_index(drop=True)
df_unamb    = df_all[~df_all.is_conflict].reset_index(drop=True)

# Load Phase 1 circuit head lists
with open(PHASE1_DIR / 'circuit_A_heads.json') as f:
    circuit_A_heads = set(map(tuple, json.load(f)))
with open(PHASE1_DIR / 'circuit_B_heads.json') as f:
    circuit_B_heads = set(map(tuple, json.load(f)))

print(f'Circuit A top-20: {sorted(circuit_A_heads)[:5]}...')
print(f'Circuit B top-20: {sorted(circuit_B_heads)[:5]}...')

## Step 1 — Activation magnitude delta (conflict vs. unambiguous)

For each prompt, compute the mean L2 norm of the `z` (value-weighted output)
vector for every head.  Then take the difference: conflict − unambiguous.

In [ ]:
# Conflict magnitudes
conflict_mags = []
for _, row in tqdm(df_conflict.iterrows(), total=len(df_conflict), desc='Conflict magnitudes'):
    tokens = model.to_tokens(row['prompt_text'])
    mag = head_activation_magnitudes(model, tokens)  # (n_layers, n_heads)
    conflict_mags.append(mag)

conflict_mags = np.stack(conflict_mags)  # (n_conflict, n_layers, n_heads)
np.save(RESULTS_DIR / 'conflict_magnitudes.npy', conflict_mags)
print(f'Conflict magnitudes: {conflict_mags.shape}')

In [ ]:
# Unambiguous magnitudes
unamb_mags = []
for _, row in tqdm(df_unamb.iterrows(), total=len(df_unamb), desc='Unambiguous magnitudes'):
    tokens = model.to_tokens(row['prompt_text'])
    mag = head_activation_magnitudes(model, tokens)
    unamb_mags.append(mag)

unamb_mags = np.stack(unamb_mags)  # (n_unamb, n_layers, n_heads)
np.save(RESULTS_DIR / 'unamb_magnitudes.npy', unamb_mags)
print(f'Unambiguous magnitudes: {unamb_mags.shape}')

In [ ]:
# Compute delta
delta = arbitration_delta_score(conflict_mags, unamb_mags)  # (n_layers, n_heads)
np.save(RESULTS_DIR / 'activation_delta.npy', delta)

ranked = rank_heads_by_delta(delta)
df_ranked = pd.DataFrame(ranked, columns=['layer', 'head', 'delta_score'])
df_ranked.to_csv(RESULTS_DIR / 'ranked_heads.csv', index=False)

print('Top 15 arbitration head candidates:')
print(f'{"Rank":>4}  {"Layer":>5}  {"Head":>4}  {"Delta":>9}  {"In Circuit A":>12}  {"In Circuit B":>12}')
for i, (l, h, s) in enumerate(ranked[:15], 1):
    in_A = '★' if (l, h) in circuit_A_heads else ''
    in_B = '★' if (l, h) in circuit_B_heads else ''
    print(f'{i:>4}  {l:>5}  {h:>4}  {s:>9.4f}  {in_A:>12}  {in_B:>12}')

### Arbitration head heatmap (all 144 heads)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
vmax = max(abs(delta.max()), abs(delta.min()))
im = ax.imshow(delta, cmap='RdYlGn', vmin=-vmax, vmax=vmax,
               aspect='auto', interpolation='nearest')

# Annotate top-5 candidates
for rank, (l, h, s) in enumerate(ranked[:5], 1):
    ax.add_patch(plt.Rectangle((h - 0.5, l - 0.5), 1, 1,
                                fill=False, edgecolor='black', lw=2.5))
    ax.text(h, l, str(rank), ha='center', va='center',
            fontsize=7, fontweight='bold', color='black')

ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_xticks(range(N_HEADS))
ax.set_yticks(range(N_LAYERS))
ax.set_title('Arbitration Head Candidates: Conflict − Unambiguous Activation Delta\n'
             '(boxes = top-5 candidates, rank labelled)')
plt.colorbar(im, ax=ax, label='Mean activation delta (conflict − unambiguous)')
plt.tight_layout()
plt.savefig('../figures/03_arbitration_head_heatmap.png', dpi=150)
plt.show()

## Step 2 — Ablation test: top-10 candidate heads

In [ ]:
TOP_K_ABL = 10
ablation_results = []

for rank, (layer, head, delta_score) in enumerate(ranked[:TOP_K_ABL], 1):
    print(f'[{rank}/{TOP_K_ABL}] Ablating L{layer}H{head} (delta={delta_score:.4f})...')
    
    result = ablation_flip_rate(
        model=model,
        prompt_rows=df_conflict,
        layer=layer,
        head=head,
    )
    result.update({
        'rank': rank,
        'layer': layer,
        'head': head,
        'delta_score': delta_score,
        'in_circuit_A': (layer, head) in circuit_A_heads,
        'in_circuit_B': (layer, head) in circuit_B_heads,
    })
    ablation_results.append(result)
    print(f'  Flip rate: {result["flip_rate"]:.1%} ({result["n_flipped"]}/{result["n_prompts"]})')

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv(RESULTS_DIR / 'ablation_results.csv', index=False)
print('\nAblation results saved.')
df_ablation[['rank', 'layer', 'head', 'delta_score', 'flip_rate', 'mean_diff_orig', 'mean_diff_abl']]

### Ablation: effect on logit-lens curve for top candidate

In [ ]:
# Pick the top candidate
top_layer, top_head, _ = ranked[0]
print(f'Top candidate: L{top_layer}H{top_head}')

# Sample a few conflict prompts and show how ablation changes the curve
sample_rows = df_conflict.head(5)
from circuit_conflict.utils import logit_lens_diff as _lld

fig, axes = plt.subplots(1, min(5, len(sample_rows)), figsize=(15, 4), sharey=False)
if len(sample_rows) == 1:
    axes = [axes]

layer_x = np.arange(N_LAYERS)
for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    try:
        tok_A = get_answer_token_id(model, row['answer_A'])
        tok_B = get_answer_token_id(model, row['answer_B'])
    except ValueError:
        continue
    tokens = model.to_tokens(row['prompt_text'])

    # Baseline curve
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
    curve_base = _lld(model, cache, tok_A, tok_B)

    # Ablated curve
    curve_abl = ablate_head_layer_curve(model, tokens, tok_A, tok_B, top_layer, top_head)

    ax.plot(layer_x, curve_base, label='Original', lw=2)
    ax.plot(layer_x, curve_abl, label=f'Ablate L{top_layer}H{top_head}', lw=2, linestyle='--')
    ax.axhline(0, color='black', lw=0.8, linestyle=':')
    ax.set_title(row['prompt_id'], fontsize=8)
    ax.set_xlabel('Layer')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Logit diff (A − B)')
plt.suptitle(f'Effect of Ablating Top Arbitration Head (L{top_layer}H{top_head})', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/03_ablation_curve_shift.png', dpi=150)
plt.show()

## Step 3 — Activation patching: losing circuit into arbitration head

In [ ]:
patch_results = []

# For each conflict prompt, find its unambiguous partner with the *losing* answer
for _, crow in tqdm(df_conflict.iterrows(), total=len(df_conflict), desc='Patching test'):
    cid = crow['prompt_id']
    uid = cid.replace('_conflict', '_unamb')
    urows = df_unamb[df_unamb.prompt_id == uid]
    if len(urows) == 0:
        continue
    urow = urows.iloc[0]

    try:
        tok_A = get_answer_token_id(model, crow['answer_A'])
        tok_B = get_answer_token_id(model, crow['answer_B'])
    except ValueError:
        continue

    tokens_conflict = model.to_tokens(crow['prompt_text'])
    tokens_unamb    = model.to_tokens(urow['prompt_text'])

    # The 'losing' circuit tokens = the unambiguous version
    # (represents the answer the model *doesn't* choose in the conflict)
    result = patch_losing_circuit_into_head(
        model, tokens_conflict, tokens_unamb,
        tok_A, tok_B,
        layer=top_layer, head=top_head,
    )
    result['prompt_id'] = cid
    result['category']  = crow['category']
    patch_results.append(result)

df_patch = pd.DataFrame(patch_results)
df_patch.to_csv(RESULTS_DIR / 'patch_results.csv', index=False)

patch_success_rate = df_patch['answer_flipped'].mean()
print(f'Patch success rate (answer flipped): {patch_success_rate:.1%} '
      f'({df_patch["answer_flipped"].sum()}/{len(df_patch)} prompts)')
print(df_patch.groupby('category')['answer_flipped'].mean())

## Step 4 — Cross-category analysis

In [ ]:
# Compute per-category activation deltas
cat_deltas = {}
for cat in ['A', 'B', 'C']:
    conf_ids = df_conflict[df_conflict.category == cat].index.tolist()
    unamb_ids = df_unamb[df_unamb.category == cat].index.tolist()
    if not conf_ids or not unamb_ids:
        continue
    cat_conflict_mag = conflict_mags[conf_ids]
    cat_unamb_mag    = unamb_mags[unamb_ids]
    cat_deltas[cat]  = arbitration_delta_score(cat_conflict_mag, cat_unamb_mag)

jaccard_df = cross_category_jaccard(cat_deltas, k=10)
jaccard_df.to_csv(RESULTS_DIR / 'jaccard_matrix.csv')
print('Cross-category Jaccard similarity (top-10 arbitration heads):')
print(jaccard_df)

In [ ]:
# Print which heads appear in multiple categories
print('\nTop-10 heads per category:')
category_top_sets = {}
for cat, d in cat_deltas.items():
    top_set = top_k_heads(d, k=10)
    category_top_sets[cat] = top_set
    print(f'  Category {cat}: {sorted(top_set)}')

# Find universal heads (in all categories)
if len(category_top_sets) == 3:
    universal = category_top_sets['A'] & category_top_sets['B'] & category_top_sets['C']
    print(f'\nUniversal arbitration heads (all 3 categories): {sorted(universal)}')
    bicat = (
        (category_top_sets.get('A', set()) & category_top_sets.get('B', set())) |
        (category_top_sets.get('A', set()) & category_top_sets.get('C', set())) |
        (category_top_sets.get('B', set()) & category_top_sets.get('C', set()))
    )
    print(f'Appear in ≥2 categories: {sorted(bicat)}')

print('\nPhase 3 complete. ✓')